# Mel-Band RoFormer — Vocal / Instrumental Separation
Google Colab GPU runtime. Source: https://github.com/openmirlab/melband-roformer-infer
Uses the default Kimberley Jensen vocal model.

In [ ]:
%pip install -q "melband-roformer-infer @ git+https://github.com/openmirlab/melband-roformer-infer.git"

In [ ]:
from google.colab import files
from pathlib import Path
import subprocess

uploaded = files.upload()
input_dir = Path('/content/music_input')
input_dir.mkdir(exist_ok=True)
for name, content in uploaded.items():
    src = Path('/content') / name
    src.write_bytes(content)
    wav = input_dir / f'{src.stem}.wav'
    subprocess.run(['ffmpeg', '-y', '-i', str(src), '-ar', '44100', '-ac', '2', str(wav)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    print(wav)

In [ ]:
from mel_band_roformer import MelBandRoformerSession

output_dir = '/content/music_stems'
with MelBandRoformerSession(device='cuda:0') as session:
    manifest = session.infer(str(input_dir), store_dir=output_dir)
for item in manifest:
    print(item['output_id'], item['output_path'])

In [ ]:
from IPython.display import Audio, display

for item in manifest:
    print(item['output_id'])
    display(Audio(filename=item['output_path']))

## 1. 곡 구조 분석 — LinkSeg (9 classes)
원본 음원에서 Verse / Pre-Chorus / Chorus 등의 시간 구간을 탐지합니다. LinkSeg 원본의 Python 3.9 환경을 별도 구성합니다.

In [ ]:
%pip -q install uv
import subprocess
from pathlib import Path
linkseg_root = Path("/content/LinkSeg")
if not linkseg_root.exists():
    subprocess.run(["git", "clone", "https://github.com/morgan76/LinkSeg.git", str(linkseg_root)], check=True)
subprocess.run(["uv", "python", "install", "3.9"], check=True)
subprocess.run(["uv", "venv", "/content/linkseg_env", "--python", "3.9"], check=True)
linkseg_python = "/content/linkseg_env/bin/python"
subprocess.run(["uv", "pip", "install", "--python", linkseg_python, "numpy==1.23.5", "Cython<3", "setuptools<70"], check=True)
subprocess.run(["uv", "pip", "install", "--python", linkseg_python, "git+https://github.com/CPJKU/madmom"], check=True)
subprocess.run(["uv", "pip", "install", "--python", linkseg_python, "-r", str(linkseg_root / "requirements.txt")], check=True)

In [ ]:
import json
import shutil
from pathlib import Path
import subprocess

# Original full-mix WAV (not the extracted vocal stem) is used for song structure.
linkseg_data = Path("/content/linkseg_data")
(linkseg_data / "audio").mkdir(parents=True, exist_ok=True)
(linkseg_data / "predictions").mkdir(parents=True, exist_ok=True)
original_wavs = sorted(Path(input_dir).glob("*.wav"))
for track in original_wavs:
    shutil.copy2(track, linkseg_data / "audio" / track.name)

subprocess.run([linkseg_python, str(linkseg_root / "src/preprocess_data.py"), "--data_path", str(linkseg_data)], check=True, cwd=str(linkseg_root / "src"))
subprocess.run([
    linkseg_python, str(linkseg_root / "src/predict.py"),
    "--test_data_path", str(linkseg_data),
    "--model_name", str(linkseg_root / "data/model_9_classes.pt"),
    "--nb_section_labels", "9",
    "--gpu", "0"
], check=True, cwd=str(linkseg_root / "src"))

sections_by_track = {}
for track in original_wavs:
    jam_path = linkseg_data / "predictions" / f"{track.stem}.jams"
    jam = json.loads(jam_path.read_text())
    annotations = [a for a in jam["annotations"] if a["namespace"].startswith("segment")]
    sections = []
    for obs in annotations[0]["data"]:
        label = obs["value"]
        if isinstance(label, dict):
            label = label.get("label", str(label))
        sections.append({
            "label": str(label),
            "start": float(obs["time"]),
            "end": float(obs["time"]) + float(obs["duration"])
        })
    sections_by_track[track.stem] = sections
    print(f"\n{track.stem}")
    for idx, section in enumerate(sections, 1):
        print(f"{idx:02d}. {section['label']:12s} {section['start']:7.2f}s – {section['end']:7.2f}s")

## 2. 구간별 보컬 멜로디 추출 — Basic Pitch
보컬 stem을 Verse / Pre-Chorus / Chorus별로 잘라 MIDI를 만듭니다. Basic Pitch 역시 별도 Python 환경에서 실행합니다.

In [ ]:
subprocess.run(["uv", "python", "install", "3.11"], check=True)
subprocess.run(["uv", "venv", "/content/basicpitch_env", "--python", "3.11"], check=True)
basicpitch_python = "/content/basicpitch_env/bin/python"
subprocess.run(["uv", "pip", "install", "--python", basicpitch_python, "basic-pitch"], check=True)
%pip -q install pretty_midi

In [ ]:
from pathlib import Path
import subprocess

target_sections = {"verse", "pre-chorus", "chorus"}
midi_root = Path("/content/section_midis")
midi_root.mkdir(parents=True, exist_ok=True)
midi_results = []

for track_stem, sections in sections_by_track.items():
    vocal_outputs = [
        item for item in manifest
        if "vocal" in str(item["output_id"]).lower()
        and track_stem.lower() in str(item["output_path"]).lower()
    ]
    if not vocal_outputs:
        raise FileNotFoundError(f"Could not match vocal stem for {track_stem}")
    vocal_path = Path(vocal_outputs[0]["output_path"])

    for i, section in enumerate(sections, 1):
        label = section["label"].lower().replace("_", "-").replace(" ", "-")
        if label not in target_sections:
            continue
        name = f"{track_stem}_{i:02d}_{label}"
        clip = midi_root / f"{name}.wav"
        subprocess.run([
            "ffmpeg", "-y", "-loglevel", "error",
            "-ss", str(section["start"]), "-to", str(section["end"]),
            "-i", str(vocal_path), str(clip)
        ], check=True)
        subprocess.run([
            basicpitch_python, "-m", "basic_pitch", str(midi_root), str(clip)
        ], check=True)
        midi_path = midi_root / f"{name}_basic_pitch.mid"
        midi_results.append((section, clip, midi_path))

for section, clip, midi_path in midi_results:
    print(section["label"], f"{section['start']:.2f}s–{section['end']:.2f}s", midi_path)

## 3. 선택한 구간의 MIDI 재생
인덱스를 선택하면 해당 구간의 추출 보컬과 MIDI 선율을 각각 재생합니다.

In [ ]:
from IPython.display import Audio, display
import pretty_midi

for i, (section, clip, midi_path) in enumerate(midi_results):
    print(i, section["label"], clip.stem)

SELECT_INDEX = 0 #@param {type:"integer"}
section, clip, midi_path = midi_results[SELECT_INDEX]
print("Vocal:", clip.name)
display(Audio(filename=str(clip)))
print("MIDI:", midi_path.name)
midi = pretty_midi.PrettyMIDI(str(midi_path))
display(Audio(midi.synthesize(fs=22050), rate=22050))